In [ ]:
import pandas as pd
import glob

# načtení všech CSV souborů jednotlivých krajů ze složky
kraje = glob.glob(
    "C:/Dominika/industrial_analysis/původní data/zamestnanci_odvetvi/csv/*.csv"
)

# spojení do jednoho DataFrame pro celou Českou republiku
zamestnanci_komplet = pd.concat([pd.read_csv(f) for f in kraje], ignore_index=True)

# odstranění posledního sloupce bez hodnot a řádků ve sloupcích Kraj, Zaměstnáni v NH, Pohlaví s prázdnou hodnotou
zamestnanci_komplet = zamestnanci_komplet.drop(columns=["Unnamed: 2"], errors="ignore")
zamestnanci_komplet = zamestnanci_komplet.dropna(
    subset=["Kraj", "ZAMĚSTNANÍ V NH", "Pohlaví"]
).copy()

# převod roků do jednoho sloupce
zamestnanci_komplet = zamestnanci_komplet.melt(
    id_vars=["Kraj", "ZAMĚSTNANÍ V NH", "Pohlaví"],
    var_name="Rok",
    value_name="Pocet_zamestnancu",
)

# 2) pivot zaměstnání do sloupců
zamestnanci_komplet = zamestnanci_komplet.pivot_table(
    index=["Kraj", "Rok", "Pohlaví"],
    columns="ZAMĚSTNANÍ V NH",
    values="Pocet_zamestnancu",
    aggfunc="sum",
).reset_index()
zamestnanci_komplet.columns.name = None

# převod roků na int
zamestnanci_komplet["Rok"] = zamestnanci_komplet["Rok"].astype(int)

# převedení znaků ".", "-", NaN na 0 ve sloupcích s odvětvími činnosti a následně převod na int
odvetvi = zamestnanci_komplet.columns.difference(["Kraj", "Rok", "Pohlaví"])
zamestnanci_komplet[odvetvi] = (
    zamestnanci_komplet[odvetvi]
    .replace([".", "-"], 0)
    .replace(",", ".", regex=True)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)
# ověření absence NaN hodnot v souboru
assert zamestnanci_komplet.isna().sum().sum() == 0
assert not zamestnanci_komplet.duplicated(["Kraj", "Rok", "Pohlaví"]).any()

# seřazení
zamestnanci_komplet = zamestnanci_komplet.sort_values(["Kraj", "Rok", "Pohlaví"])


zamestnanci_komplet
# zamestnanci_komplet.to_csv("zamestnanci_odvetvi.csv", index=False)

,Kraj,Rok,Pohlaví,Administrativní a podpůrné činnosti,Doprava a skladování,Informační a komunikační činnosti,"Kulturní, zábavní a rekreační činnosti",Ostatní činnosti,Peněžnictví a pojišťovnictví,"Profesní, vědecké a technické činnosti",...,"Ubytování, stravování a pohostinství",Velkoobchod a maloob.; opr. mot. vozidel,Veřejná správa a obrana; pov. soc. zabezp.,Vzdělávání,"Výroba a rozvod elektřiny, plynu, tepla",Zdravotní a sociální péče,"Zemědělství, lesnictví a rybářství",Zpracovatelský průmysl,Zásob. vodou; činnosti souvis. s odpady,Činnosti v oblasti nemovitostí
0,Jihomoravský kraj,1993,M,7157,20834,5098,5315,2799,2034,10958,...,6331,19487,18699,15160,6123,6505,29286,94721,2008,1840
1,Jihomoravský kraj,1993,Z,5021,9472,6405,3492,6086,3985,10957,...,9142,24961,12152,26497,2115,21081,17095,64421,629,2726
2,Jihomoravský kraj,1994,M,7485,23957,4937,3742,2047,2148,12819,...,5525,23720,21797,15615,6734,7994,21807,90076,3026,1230
3,Jihomoravský kraj,1994,Z,4711,7792,5845,4114,5642,6079,9217,...,8110,29509,12167,27660,1911,24876,13714,64160,1052,2788
4,Jihomoravský kraj,1995,M,8186,23944,6520,3298,3328,2630,14471,...,6367,27128,21527,13420,5873,5737,18564,85296,2515,1238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,Ústecký kraj,2022,Z,3602,7038,1574,3197,5616,3101,5865,...,7705,24260,13447,20659,857,25555,2057,32873,2423,926
892,Ústecký kraj,2023,M,4447,22738,4118,4472,1561,2348,6842,...,6041,16625,11702,4325,7226,6569,4173,59413,2942,1384
893,Ústecký kraj,2023,Z,5417,7803,1061,2581,3952,3292,6991,...,7691,17656,10335,22058,837,26618,1833,33247,940,1797
894,Ústecký kraj,2024,M,3994,22345,3837,2259,1322,1940,5692,...,4598,15536,15798,4336,6176,6614,6004,58810,5078,1599
